# Double-Well Branching Demonstration

This notebook explores a signed-particle branching model in a quartic
double-well potential.

**Scientific caution:** the branching rate and momentum shifts used here
are heuristic. They are not derived from the exact Wigner kernel, so this
notebook should be treated as a numerical experiment rather than a
quantitatively validated tunneling solver.

## Before running

From the repository's top-level folder:

```bash
python3 -m pip install -e .
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mcw import (
    sample_wigner_gaussian,
    gaussian_kde_marginal,
    double_well_potential,
    propagate_double_well_heuristic,
)


In [ ]:
a = 0.01
well_position = 4.0
x = np.linspace(-40.0, 40.0, 2**12, endpoint=False)

potential = double_well_potential(
    x,
    a=a,
    x0=well_position,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, potential)
ax.set_xlim(-12, 12)
ax.set_ylim(
    0,
    1.3 * double_well_potential(
        0.0,
        a=a,
        x0=well_position,
    ),
)
ax.set_xlabel("x")
ax.set_ylabel("V(x)")
ax.set_title("Quartic double-well potential")
fig.tight_layout()
plt.show()


In [ ]:
dt = 0.02
nsteps = 200
steps = {0, nsteps // 2, nsteps}

xs0, ps0, signs0 = sample_wigner_gaussian(
    5_000,
    x0=-well_position,
    p0=0.0,
    sigma=1.0,
    seed=7,
)

snapshots = propagate_double_well_heuristic(
    xs0,
    ps0,
    signs0,
    dt,
    nsteps,
    a=a,
    x0=well_position,
    gamma0=0.01,
    momentum_shift=0.7,
    seed=7,
    snapshot_steps=steps,
)


In [ ]:
for step in sorted(steps):
    xs, _, signs = snapshots[step]
    density = gaussian_kde_marginal(
        xs,
        signs,
        x,
        bandwidth=0.3,
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(x, density)
    ax.set_xlim(-12, 12)
    ax.set_xlabel("x")
    ax.set_ylabel("Signed marginal estimate")
    ax.set_title(
        f"Heuristic branching, t={step * dt:.2f}, "
        f"particles={len(xs):,}"
    )
    fig.tight_layout()
    plt.show()
